# Parkinson's Disease Detection — Train SVM + 4 Baselines

Reproducibility notebook for the
[`Parkinsons-Disease-Detection-SVM`](https://github.com/MatamDinesh0802/Parkinsons-Disease-Detection-SVM) project.

**This notebook runs in seconds on CPU** — no GPU required. You can run it
locally inside the project's `.venv`, or open it in [Google Colab](https://colab.research.google.com/)
and it will fetch the dataset directly from UCI.

It will:

1. Load the **UCI Parkinson's** dataset (195 voice recordings, 22 features).
2. Run EDA — class balance, feature distributions, correlation heatmap.
3. Train 5 models: SVM (linear), SVM (RBF), Logistic Regression, Random Forest, Gradient Boosting.
4. Pick the best by **ROC-AUC** on a held-out test set.
5. Save `best_model.joblib` + `scaler.joblib` + `metrics.json` + confusion matrix + ROC curves.

---

## 1. Setup

In [ ]:
# Only need to run on Colab — locally these are already in .venv
!pip install -q numpy pandas scikit-learn matplotlib seaborn joblib

In [ ]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

## 2. Load the UCI Parkinson dataset

In [ ]:
# If running locally, prefer the bundled CSV; otherwise fetch from UCI mirror
local_path = Path('../data/raw/parkinsons.csv')
if local_path.exists():
    df = pd.read_csv(local_path)
    print(f'Loaded local CSV ({len(df)} rows)')
else:
    UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data'
    df = pd.read_csv(UCI_URL)
    print(f'Fetched from UCI ({len(df)} rows)')

df.head()

In [ ]:
FEATURE_COLUMNS = [c for c in df.columns if c not in ('name', 'status')]
TARGET = 'status'

print(f'Features: {len(FEATURE_COLUMNS)}')
print(f'Class balance:')
print(df[TARGET].value_counts(normalize=True).rename({0: 'Healthy', 1: "Parkinson's"}))

## 3. EDA — class balance, feature distributions, correlation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

counts = df[TARGET].value_counts().rename({0: 'Healthy', 1: "Parkinson's"})
axes[0].bar(counts.index, counts.values, color=['#10B981', '#F43F5E'])
axes[0].set_title('Class distribution')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2, str(v), ha='center')

df.boxplot(column='MDVP:Fo(Hz)', by=TARGET, ax=axes[1])
axes[1].set_title('Fundamental frequency by class')
axes[1].set_xticklabels(['Healthy', "Parkinson's"])
plt.suptitle('')
fig.tight_layout()
fig.savefig(FIG_DIR / 'eda_overview.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(11, 9))
corr = df[FEATURE_COLUMNS + [TARGET]].corr()
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
            cbar_kws={'shrink': 0.7}, xticklabels=True, yticklabels=True)
ax.set_title('Feature correlation (incl. status)', fontsize=12)
plt.setp(ax.get_xticklabels(), rotation=60, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()

## 4. Train / test split + standardize

In [ ]:
X = df[FEATURE_COLUMNS].values
y = df[TARGET].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE,
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'train {X_train_s.shape}  test {X_test_s.shape}')

## 5. Train 5 models

In [ ]:
models = {
    'svm_linear':         SVC(kernel='linear', probability=True, random_state=RANDOM_STATE),
    'svm_rbf':            SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE),
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'random_forest':      RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    'gradient_boosting':  GradientBoostingClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=RANDOM_STATE),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_s, y_train)
    y_pred = clf.predict(X_test_s)
    y_proba = clf.predict_proba(X_test_s)[:, 1]
    results[name] = {
        'y_pred': y_pred, 'y_proba': y_proba,
        'metrics': {
            'accuracy':  float(accuracy_score(y_test, y_pred)),
            'precision': float(precision_score(y_test, y_pred)),
            'recall':    float(recall_score(y_test, y_pred)),
            'f1':        float(f1_score(y_test, y_pred)),
            'roc_auc':   float(roc_auc_score(y_test, y_proba)),
        },
        'report': classification_report(y_test, y_pred, output_dict=True),
    }
    m = results[name]['metrics']
    print(f"  {name:22s} acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  auc={m['roc_auc']:.4f}")

## 6. Confusion matrices + ROC curves + model comparison

In [ ]:
for name, r in results.items():
    cm = confusion_matrix(y_test, r['y_pred'])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Healthy', "Parkinson's"],
                yticklabels=['Healthy', "Parkinson's"],
                cbar=False, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(f'Confusion matrix — {name}')
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'confusion_matrix_{name}.png', dpi=150)
    plt.close(fig)

# ROC curves
fig, ax = plt.subplots(figsize=(6.5, 5))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={r['metrics']['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('ROC — model comparison'); ax.legend(loc='lower right', fontsize=9)
fig.tight_layout(); fig.savefig(FIG_DIR / 'roc_curves.png', dpi=150)
plt.show()

# Bar chart comparing metrics
names = list(results.keys())
metric_keys = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(names))
width = 0.16
for i, k in enumerate(metric_keys):
    vals = [results[n]['metrics'][k] for n in names]
    ax.bar(x + i*width, vals, width, label=k)
ax.set_xticks(x + 2*width)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.set_ylim(0, 1.05); ax.set_title('Model comparison')
ax.legend(fontsize=9, ncol=5, loc='lower right')
fig.tight_layout(); fig.savefig(FIG_DIR / 'model_comparison.png', dpi=150)
plt.show()

## 7. Save artifacts

In [ ]:
best_name = max(results, key=lambda n: results[n]['metrics']['roc_auc'])
print(f'Best model by ROC-AUC: {best_name}')

joblib.dump(models[best_name], OUT_DIR / 'best_model.joblib')
joblib.dump(scaler, OUT_DIR / 'scaler.joblib')
(OUT_DIR / 'best_model_name.txt').write_text(best_name)

metrics_out = {
    'best_model': best_name,
    'n_train': int(len(y_train)), 'n_test': int(len(y_test)),
    'models': {n: r['metrics'] for n, r in results.items()},
}
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print('Saved:', sorted(os.listdir(OUT_DIR)))

## 8. Download (Colab) — drop these into the repo

| Colab file | Destination in the repo |
|---|---|
| `outputs/best_model.joblib` | `models/best_model.joblib` |
| `outputs/scaler.joblib` | `models/scaler.joblib` |
| `outputs/best_model_name.txt` | `models/best_model_name.txt` |
| `outputs/metrics.json` | `reports/metrics.json` |
| `outputs/figures/*.png` | `reports/figures/` |


In [ ]:
import shutil
shutil.make_archive('parkinsons_artifacts', 'zip', OUT_DIR)
try:
    from google.colab import files
    files.download('parkinsons_artifacts.zip')
except ImportError:
    print('Not on Colab — artifacts are in outputs/.')